In [ ]:
#after 
# train test split
class MnistDataset(Dataset):

    def __init__(self,
                 dataframe:pd.DataFrame,
                 device:str=device,
                 label_col=None,
                 transformer=None):
        super(MnistDataset,self).__init__()

        self.df=dataframe
        self.device=device

        self.label_col= label_col
        self.transformer= transformer


        self.features=self.df.drop('label',axis=1).to_numpy()
        self.features=self.features/255
        self.labels=self.df['label'].to_numpy()



    def __len__(self):
        return len(self.features)

    def __getitem__(self,index):
        features=self.features[index]
        labels=self.labels[index]
        features= torch.tensor(features, dtype=torch.float32)
        labels= torch.tensor(labels, dtype=torch.int64)    #int64?
        return features,labels
    
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, test_loss):
        if self.best_loss is None:
            self.best_loss = test_loss
        elif test_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = test_loss
            self.counter = 0
train_ds= MnistDataset(dataframe=data_df, label_col=labels)
test_ds=MnistDataset(dataframe=test_df, label_col=labels)

#dataloader shuffles the data
train_loader=DataLoader(dataset=train_ds,batch_size=BATCH_SIZE, shuffle=True)
test_loader=DataLoader(dataset=test_ds,batch_size=BATCH_SIZE, shuffle=True)


# Loss Function FOR MULTI-CLASS outputs
loss_fn = nn.CrossEntropyLoss()

# Optimizers
optimizer = torch.optim.Adam(model.parameters(), lr = ALPHA, weight_decay=1e-4)

#early stopping
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

# Some lists to collect data
loss = []
tloss = []
n_epoch = []
acc = []
tacc = []

# Iterations
for epoch in range (EPOCHS):

    model.train() # Set the model in training mode

    epoch_loss = 0
    epoch_acc = 0
    tepoch_loss= 0
    tepoch_acc = 0


    for batch_idx, (train_X, train_y) in enumerate(train_loader):

        train_X, train_y = train_X.to(device), train_y.to(device)

        predict_prob = model(train_X) # make predictions

        batch_loss = loss_fn(predict_prob, train_y) # calculate loss

        epoch_loss += (batch_loss - epoch_loss)/(batch_idx+1)

        ###---------------
        ### Back prop step
        ###---------------
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        _, y_pred = torch.max(predict_prob, 1) # get predicted class from the probabilities

        batch_acc = accuracy_score( train_y.cpu().numpy(), y_pred.data.cpu()) # move to CPU

        epoch_acc += (batch_acc - epoch_acc)/(batch_idx+1)

    loss.append ( epoch_loss.data.item() ) # append to loss list
    acc.append(epoch_acc) # append to accuracy list

    model.eval() # evaluation mode, prevent from learning

    with torch.inference_mode():
        for batch_idx, (test_X, test_y) in enumerate(test_loader):

            test_X, test_y =test_X.to(device), test_y.to(device)

            predict_prob_tst = model(test_X) # make predictions on test data

            tbach_loss = loss_fn( predict_prob_tst, test_y)

            tepoch_loss += (tbach_loss - tepoch_loss)/(batch_idx+1)

            _, y_pred = torch.max(predict_prob_tst, 1) # get predicted class from the probabilities

            tbatch_acc = accuracy_score( test_y.cpu().numpy(), y_pred.data.cpu()) # move to CPU

            tepoch_acc += (tbatch_acc - tepoch_acc) / (batch_idx+1)

        tacc.append(tepoch_acc) # append to accuracy list

        tloss.append( tepoch_loss.data.item())  # append to loss list

    n_epoch.append(epoch)


    if epoch % 20 == 0:
        fmtStr = 'Epoch :{:5d}/{:5d} --- Loss: {:.5f}/{:.5f} | Acc {:.5f}/{:.5f}'

        print (fmtStr.format(epoch, EPOCHS,
                             epoch_loss.data.item(),
                             tepoch_loss.data.item(),
                             epoch_acc,
                             tepoch_acc))

    # Check Early Stopping
    early_stopping(tepoch_loss.item()) # Removed the 'model' argument
    if early_stopping.early_stop:
        print("Early stopping triggered. Halting training.")

        # The EarlyStopping class does not currently store best_weights.
        # To properly restore the best model, the EarlyStopping class
        # (in cell `5619bdb4-122b-454a-86ae-bbac46c23632`) needs to be modified
        # to save model.state_dict() when the best loss improves.
        # model.load_state_dict(early_stopping.best_weights) # Commented out to prevent AttributeError
        break

    loss_df = pd.DataFrame({'epoch': n_epoch,
                       'loss' : loss,
                       'test_loss' : tloss,
                       'acc' : acc,
                       'test_acc': tacc})
    
    # Train Data prediction
y_pred = []
y_true = []

model.eval()

with torch.inference_mode():
    for batch_idx, (train_X, train_y) in enumerate(train_loader):
        train_X = train_X.to(device)

        outputs = model(train_X)

        y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        y_true.extend(train_y.cpu().numpy())

fn_plot_confusion_matrix(y_true, y_pred, labels=class_names)

In [ ]:
#for regression
# Compile the model with a specific optimizer, loss function, and metrics

model.compile(optimizer='adam', 
              loss='mean_squared_error', 
              metrics=['mae'])

# fit the model
history = model.fit(X_train, y_train, 
                    batch_size = BATCH_SIZE,
                    validation_data=[X_test, y_test],
                    epochs=EPOCHS, 
                    verbose=2)

hist_df = pd.DataFrame(history.history)

fn_plot_tf_hist(hist_df)

# evaluate the model
error = model.evaluate(X_test, y_test, verbose=0)

print(f'Loss: {error[0]:.3f}, MAE: {np.sqrt(error[1]):.3f}')

y_pred = model.predict(X_test)

results_df = pd.DataFrame({'pred' : y_pred[:,0], 'test' : y_test} )

results_df['y_true'] = y_true_test

ax = results_df.pred.plot(c = 'r', label = 'pred', lw = 4);
results_df.y_true.plot(c = 'k', ax = ax, label = 'true')

ax.scatter(results_df.index, results_df.test, c='DarkBlue', marker ='*', label = 'test');

ax.set_xlabel("epoch")
ax.set_ylabel("Price")
ax.grid()
plt.legend()
plt.show()



In [ ]:
#regression with pytorh

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset





inpDir = '../../input' # location where input data is stored
outDir = '../output' # location to store outputs

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible  results
torch.manual_seed(RANDOM_STATE) # setting for PyTorch as well

EPOCHS = 51
ALPHA = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32
# Set parameters for decoration of plots
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

CMAP = plt.cm.coolwarm

plt.rcParams.update(params) # update rcParams

# Prepare data loaders
X_train_tensor = torch.FloatTensor(X_train).reshape(-1, 1)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class Swish(nn.Module):
    """Swish activation function: x * sigmoid(x)"""
    def forward(self, x):
        return x * torch.sigmoid(x)
    
class RegressionMLP(nn.Module):

    """MLP for regression with ReLU activation"""
    
    def __init__(self, input_dim=1, hidden_dims=[128, 64]):

        super(RegressionMLP, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(Swish())
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)
    
def train_model(model, 
                train_loader, 
                test_loader, 
                epochs=EPOCHS, 
                lr=ALPHA):
    """Train a PyTorch model and return history"""
    loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    
    history = {'epoch': [],
               'train_loss': [],
               'test_loss': [],
               'train_err': [],
               'test_err': []}
    
    for epoch in range(epochs):
        history['epoch'].append(epoch)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_err = 0.0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            batch_loss = loss_fn(outputs.squeeze(), batch_y)
            batch_loss.backward()
            optimizer.step()
            
            # Detach before converting to numpy
            batch_err = mean_squared_error(
                batch_y.detach().cpu().numpy(), 
                outputs.detach().cpu().numpy()
            )
            train_loss += batch_loss.item() * batch_X.size(0)
            train_err += batch_err * batch_X.size(0)
        
        train_loss /= len(train_loader.dataset)
        train_err /= len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        history['train_err'].append(train_err)

        # Validation phase
        model.eval()
        test_loss = 0.0
        test_err = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                
                # No need to detach since no_grad already prevents gradient tracking
                batch_err = mean_squared_error(
                    batch_y.cpu().numpy(), 
                    outputs.cpu().numpy()
                )
                test_loss += loss.item() * batch_X.size(0)
                test_err += batch_err * batch_X.size(0)
        
        test_loss /= len(test_loader.dataset)
        test_err /= len(test_loader.dataset)
        history['test_loss'].append(test_loss)
        history['test_err'].append(test_err)

        if epoch % 10 == 0:
            print(f'Epoch [{epoch}/{epochs}], Loss: {train_loss:.4f} / {test_loss:.4f} | Train Err: {train_err:.4f} / Test Err: {test_err:.4f}')
    
    return history




    model = RegressionMLP(input_dim=1, 
                      hidden_dims=[128, 64]
                      ).to(device)



history = train_model(model, 
                           train_loader, 
                           test_loader, 
                           epochs=EPOCHS)


hist_df = pd.DataFrame(history)

fn_plot_torch_hist(hist_df)


# Evaluate Model 1
model.eval()
with torch.inference_mode():
    y_pred = model(X_test_tensor.to(device)).cpu().numpy()
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f'Loss (MSE): {mse:.3f}, RMSE: {rmse:.3f}')


 # Plot predictions

results_df = pd.DataFrame({
    'pred': y_pred[:, 0],
    'test': y_test,
    'y_true': y_true_test
})

fig, ax = plt.subplots(figsize=(15, 6))
results_df['pred'].plot(c='r', label='Prediction', ax=ax)
results_df['y_true'].plot(c='k', label='True', ax=ax)
plt.scatter(results_df.index, results_df['test'], c='darkblue', marker='*', label='Test')
ax.set_xlabel('Sample')
ax.set_ylabel('Value')
ax.grid()
plt.legend()
plt.title('Predictions vs True Values')
plt.show()

In [ ]:
#tensorflow

# TensorFlow Dataset Pipeline Benefits:

# from_tensor_slices() - Creates dataset from in-memory data

train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test,y_test))
## Optimize for performance

train_ds = train_ds.cache()  # Cache first
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
# shuffle() - Randomizes order (prevents overfitting to sequence)
# batch() - Groups samples for efficient processing
# Shuffle and batch the dataset
train_ds = train_ds.shuffle(buffer_size=X_train.shape[0]).batch(BATCH_SIZE)
test_ds = test_ds.shuffle(buffer_size=X_test.shape[0]).batch(BATCH_SIZE)
# cache() - Stores dataset in memory after first epoch
# prefetch() - Overlaps data preprocessing with model execution
    
inputs = tf.keras.Input(shape=(33,))

x = normalizer(inputs)
    
x = tf.keras.layers.Dense(16, activation=tf.nn.relu)(x)

x = tf.keras.layers.Dense(8, activation=tf.nn.relu)(x)

outputs = tf.keras.layers.Dense(4)(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)


loss_fn = tf.keras.losses.SparseCategoricalCrossentropy ( from_logits = True)

model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])


history = model.fit(train_ds, 
                    validation_data=test_ds,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS)

model.evaluate ( train_ds, verbose=2)

model.evaluate ( test_ds, verbose=2)

loss_df = pd.DataFrame(history.history)
loss_df.head()

fn_plot_tf_hist(loss_df)

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        first_layer = layer
        break
    
# Get weights and biases
weights = first_layer.get_weights()[0]  # Shape: (33, 18)
biases = first_layer.get_weights()[1]   # Shape: (18,)
feature_importance = np.abs(weights).mean(axis=1)

feature_names = data_df.columns[1:-1] # Exclude 'Position' and 'Position_encoded'

top_features = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots() # figsize=(10, 6)
ax.barh(top_features['feature'], top_features['importance'])
ax.set_xlabel('Average Absolute Weight')
ax.set_title('Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

y_true, y_pred = [],[]
for feats, lbls in train_ds:
    pred = probability_model(feats).numpy()
    pred = pred.argmax(axis =1)
    y_pred.extend(pred)
    y_true.extend(lbls)
print('Accuracy score on Test Data :{:.5f}'.format(accuracy_score(y_true,y_pred)))

print(classification_report( y_true, y_pred))

fn_plot_confusion_matrix( y_true, y_pred, labels=class_labels)

y_true, y_pred = [],[]
for feats, lbls in test_ds:
    pred = probability_model(feats).numpy()
    pred = pred.argmax(axis =1)
    y_pred.extend(pred)
    y_true.extend(lbls)
len(y_pred), len(y_true)

print('Accuracy score on Test Data :{:.5f}'.format(accuracy_score(y_true,y_pred)))

print(classification_report(y_true, y_pred))

fn_plot_confusion_matrix( y_true, y_pred, labels=class_labels)

In [ ]:
#ionospehere
X = data_df.drop(data_df.columns[-1], axis = 1).to_numpy()

y = data_df[data_df.columns[-1]].to_numpy()

X_train, X_test, y_train, y_test = train_test_split( X, y,
                                                   train_size = TRAIN_SIZE,
                                                   stratify=y,
                                                   random_state=RANDOM_STATE)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

le = LabelEncoder()

y_train = le.fit_transform(y_train)

y_test = le.transform(y_test)

sc = StandardScaler()

X_train = sc.fit_transform(X_train)

X_test = sc.transform(X_test)

train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test,y_test))

## Optimize for performance

train_ds = train_ds.cache()  # Cache first
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Shuffle and batch the dataset
train_ds = train_ds.shuffle(buffer_size=X_train.shape[0]).batch(BATCH_SIZE)
test_ds = test_ds.shuffle(buffer_size=X_test.shape[0]).batch(BATCH_SIZE)

## Some model functions

initalizer = tf.keras.initializers.GlorotUniform(seed = RANDOM_STATE)

optimizer = tf.keras.optimizers.Adam(learning_rate=ALPHA) # using Adam optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy( from_logits = True)

model1 = tf.keras.models.Sequential([
    
    tf.keras.layers.InputLayer( shape =(34,)  ),
    
    tf.keras.layers.Dense (26,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer),
    
    tf.keras.layers.Dense (18,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer),

   tf.keras.layers.Dense (10,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer),
    
    tf.keras.layers.Dense (2),
    
])

model1.compile(optimizer=optimizer,
               loss = loss_fn,
               metrics= ['accuracy'])


history = model1.fit( train_ds,
                    epochs=EPOCHS,
                    batch_size = BATCH_SIZE,
                    validation_data=test_ds,
                    verbose = 2)
hist_df = pd.DataFrame(history.history)
hist_df.head()

fn_plot_tf_hist(hist_df)

#with regularization
## Some model functions

initalizer = tf.keras.initializers.GlorotUniform(seed = RANDOM_STATE)

optimizer = tf.keras.optimizers.Adam(learning_rate=ALPHA) # using Adam optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy( from_logits = True)

regularizer = tf.keras.regularizers.L2(0.05)

model2 = tf.keras.models.Sequential([
    
    tf.keras.layers.InputLayer( shape =( 34, ) ),

    tf.keras.layers.Dense (26,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer,
                           kernel_regularizer = regularizer),   # ---- add L2 Regularizer
    
    tf.keras.layers.Dense (18,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer,
                           kernel_regularizer = regularizer),   # ---- add L2 Regularizer

   tf.keras.layers.Dense (10,
                           activation = tf.keras.activations.relu,
                           kernel_initializer = initalizer,
                           kernel_regularizer = regularizer),   # ---- add L2 Regularizer
    
    tf.keras.layers.Dense (2),
    
])

model2.compile(optimizer=optimizer,
               loss = loss_fn,
               metrics= ['accuracy'])


history = model2.fit(train_ds,
                    epochs=EPOCHS,
                    batch_size = BATCH_SIZE,
                    validation_data=test_ds,
                    verbose = 2)

hist_df = pd.DataFrame(history.history)


rate1 = 0.2
rate2 = 0.3
rate3 = 0.4

## Some model functions

initalizer = tf.keras.initializers.GlorotUniform(seed = RANDOM_STATE)

optimizer = tf.keras.optimizers.Adam(learning_rate=ALPHA) # using Adam optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy( from_logits = True)

model4 = tf.keras.models.Sequential([
    
    tf.keras.layers.InputLayer( shape =(34,)  ),
    
    tf.keras.layers.Dense (26,
                           activation = 'relu',
                           kernel_initializer = initalizer),
    
    tf.keras.layers.Dropout(rate1 , seed = RANDOM_STATE),
    
    tf.keras.layers.Dense (18,
                           activation = 'relu',
                           kernel_initializer = initalizer),
    
    tf.keras.layers.Dropout(rate2 , seed = RANDOM_STATE),

    tf.keras.layers.Dense (10,
                           activation = 'relu',
                           kernel_initializer = initalizer),
    
    tf.keras.layers.Dropout(rate3 , seed = RANDOM_STATE),
    
    tf.keras.layers.Dense (2),
    
])


model4.compile(optimizer=optimizer,
               loss = loss_fn,
               metrics= ['accuracy'])


history = model4.fit(train_ds,
                     epochs=EPOCHS,
                     batch_size = BATCH_SIZE,
                     validation_data=test_ds,
                     verbose = 2)

hist_df = pd.DataFrame(history.history)


In [ ]:
#feed forward neural network

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Prepare and Preprocess the Data
# Load the wine dataset (13 numerical features, 3 classes)
wine_data = load_wine()
X = wine_data.data
y = wine_data.target

# Split the data into training and holdout test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Value Normalization: Feature-wise normalization is crucial for heterogeneous data 
# so that each feature has a mean of 0 and a standard deviation of 1.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. Define the Network Architecture
# A feedforward network (multilayer perceptron) using a stack of Dense layers with ReLU activations.
# The output layer uses Softmax because this is a single-label multiclass classification problem.
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[6],)),
    layers.Dropout(0.2), # Dropout added to randomly ignore a fraction of neurons to prevent overfitting
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax") # 3 units for the 3 wine classes
])

# 3. Configure the Learning Process
# We use 'sparse_categorical_crossentropy' because our labels are provided as integers (0, 1, 2)
# rather than one-hot encoded vectors.
model.compile(optimizer="rmsprop", 
              loss="sparse_categorical_crossentropy", 
              metrics=["accuracy"])

# 4. Train the Model
# The model iterates over the training data in mini-batches. 
# We reserve 20% of the training data as a validation set to monitor performance during training.
history = model.fit(X_train, y_train,
                    epochs=40,
                    batch_size=16,
                    validation_split=0.2)

# 5. Evaluate on Test Data
# Finally, we test the model on the holdout test set to evaluate final generalization.
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

In [ ]:
#wine dataset tensorflow

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, 
                             ConfusionMatrixDisplay, f1_score)

import tensorflow as tf

labels = data_df[data_df.columns[-1]]
features_df = data_df.drop(data_df.columns[-1], axis =1)
assert features_df.shape[0] == labels.shape[0], 'Number of examples not same'

RANDOM_STATE = 24
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

EPOCHS = 100
TEST_SIZE = 0.2
ALPHA=0.001


#  Split the data in training and test sets to measure performance of the model.
X_train, X_test, y_train, y_test = train_test_split(features_df, labels, 
                                                    test_size=TEST_SIZE,
                                                    stratify=labels,
                                                    random_state=RANDOM_STATE )

print (X_train.shape, y_train.shape, X_test.shape, y_test.shape)

sc = StandardScaler()

X_train = sc.fit_transform(X_train)

X_test = sc.transform(X_test)

le = LabelEncoder() # convert 1, 2, 3 -> 0, 1, 2
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


# nodes : 13, 8, 3

model = tf.keras.Sequential([
    tf.keras.Input(shape = (X_train.shape[1],)),

    tf.keras.layers.Dense(8, activation='relu'),

    tf.keras.layers.Dense(3)
])






# inputs = tf.keras.Input(shape = (X_train.shape[1],))
    
# x = tf.keras.layers.Dense(8, activation=tf.nn.relu)(inputs)

# outputs = tf.keras.layers.Dense(3)(x)

# model1 = tf.keras.Model(inputs=inputs, outputs=outputs)

predictions = model(X_train[:1]).numpy()
predictions

model.summary()

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy ( from_logits = True)
# Step 2: Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=ALPHA)

# Step 3: Compile
model.compile(optimizer=optimizer,
              loss=loss_fn,
              metrics=['accuracy'])

history = model.fit(X_train, y_train, 
                    validation_data=[X_test, y_test],
                    epochs=EPOCHS)

model.evaluate ( X_test,  y_test, verbose=2)

loss_df = pd.DataFrame(history.history)
loss_df.head()

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred.argmax(axis=1))
print(accuracy)




def fn_plot_tf_hist(hist_df):
    
    '''
    Args:
        hist_df: a DataFrame with column names accuracy,loss	val_accuracy	val_loss
    '''
        
    fig, axes = plt.subplots(1,2 , figsize = (15,6))

    # properties  matplotlib.patch.Patch 
    props = dict(boxstyle='round', facecolor='aqua', alpha=0.4)
    facecolor = 'cyan'
    fontsize=12
    CMAP = plt.cm.coolwarm
    
    # Get columns by index to eliminate any column naming error
    y1 = hist_df.columns[0]
    y2 = hist_df.columns[1]
    y3 = hist_df.columns[2]
    y4 = hist_df.columns[3]

    # Where was min loss
    best = hist_df[hist_df[y4] == hist_df[y4].min()]
    
    ax = axes[0]

    hist_df.plot(y = [y2,y4], ax = ax, colormap=CMAP)


    # little beautification
    txtFmt = "Loss: \n  train: {:6.4f}\n   test: {:6.4f}"
    txtstr = txtFmt.format(hist_df.iloc[-1][y2],
                           hist_df.iloc[-1][y4]) #text to plot
    
    # place a text box in upper middle in axes coords
    ax.text(0.3, 0.95, txtstr, transform=ax.transAxes, fontsize=fontsize,
            verticalalignment='top', bbox=props)
    
    # calculate offset for arrow
    y_min = min(hist_df[y2].min(), hist_df[y4].min())
    y_max = max(hist_df[y2].max(), hist_df[y4].max())
    offset = (y_max-y_min)/10.0
    
    best_idx = best.index.to_numpy()[0]  # Extract first (and only) element as scalar
    best_value = best[y4].to_numpy()[0]   # Extract scalar value
    
    # Mark arrow at lowest
    ax.annotate(f'Min: {best_value:6.4f}', # text to print
                xy=(best_idx, best_value), # Arrow start
                xytext=(best_idx, best_value + offset), # location of text 
                fontsize=fontsize, va='bottom', ha='right',bbox=props, # beautification of text
                arrowprops=dict(facecolor=facecolor, shrink=0.05)) # arrow
    
    # Draw vertical line at best value
    ax.axvline(x = best_idx, color = 'green', linestyle='-.', lw = 3)

    ax.set_xlabel("Epochs")
    ax.set_ylabel(y2.capitalize())
    ax.set_title('Loss Curve')
    ax.grid(True)
    ax.legend(loc = 'upper left') # model legend to upper left
    
    ax = axes[1]

    hist_df.plot( y = [y1, y3], ax = ax, colormap=CMAP)
    
    # little beautification
    txtFmt = "Accuracy: \n  train: {:6.4f}\n  test:  {:6.4f}"
    txtstr = txtFmt.format(hist_df.iloc[-1][y1],
                           hist_df.iloc[-1][y3]) #text to plot

    # place a text box in upper middle in axes coords
    ax.text(0.3, 0.2, txtstr, transform=ax.transAxes, fontsize=fontsize,
            verticalalignment='top', bbox=props)

    # calculate offset for arroe
    y_min = min(hist_df[y1].min(), hist_df[y3].min())
    y_max = max(hist_df[y1].max(), hist_df[y3].max())
    offset = (y_max-y_min)/10.0

    best_value = best[y3].to_numpy()[0]   # Extract scalar value
    # Mark arrow at lowest
    ax.annotate(f'Best: {best_value:6.4f}', # text to print
                xy=(best_idx, best_value), # Arrow start
                xytext=(best_idx, best_value-offset), # location of text 
                fontsize=fontsize, va='bottom', ha='right',bbox=props, # beautification of text
                arrowprops=dict(facecolor=facecolor, shrink=0.05)) # arrow
    
    
    # Draw vertical line at best value
    ax.axvline(x = best_idx, color = 'green', linestyle='-.', lw = 3)

    ax.set_xlabel("Epochs")
    ax.set_ylabel(y1.capitalize())
    ax.set_title('Accuracy Curve')
    ax.grid(True)
    ax.legend(loc = 'lower left')
    
    plt.tight_layout()
    


fn_plot_tf_hist(loss_df)

loss_df[loss_df['val_loss'] == loss_df['val_loss'].min()]

probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

y_pred = probability_model(X_train).numpy()
y_pred.shape



accuracy_score(y_train, y_pred.argmax(axis=1))



print(classification_report( y_train, y_pred.argmax( axis = 1)))

y_pred = probability_model(X_test).numpy()

print('Accuracy score on Test Data :{:.5f}'.format(accuracy_score(y_test, 
                                                                  np.argmax(y_pred, axis = 1))))

print(classification_report( y_test, y_pred.argmax( axis = 1)))

def fn_plot_confusion_matrix(y_true, y_pred, labels):
    '''
    Args:
        y_true: Ground Truth 
        y_pred : Predictions
        labels : dictonary
    
    '''
    
    cm  = confusion_matrix(y_true, y_pred)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=labels.values())
    
    fig, ax = plt.subplots(figsize = (4,4))
    
    disp.plot(ax = ax, cmap = 'Blues', xticks_rotation = 'vertical', colorbar=False)
    # Disable the grid
    ax.grid(False)
    ax.set_title(f"F1_Score: {f1_score(y_test,y_pred, average='weighted'):0.4f}")
    plt.show()

    class_labels = {0: 1, 1:2, 2:3}
fn_plot_confusion_matrix( y_test, y_pred.argmax( axis = 1), labels=class_labels)



In [ ]:
#wine with pytorch
# Lets import some libraries
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torchsummary import summary



RANDOM_STATE = 24
np.random.seed(RANDOM_STATE) 
torch.manual_seed(RANDOM_STATE)

EPOCHS = 1000  

# Not needed as 'adam' has defaulted
# ALPHA = 0.001  # learning rate

TEST_SIZE = 0.2 # What fraction we want to keep for testing

# labels: The target dataset (dependent variable)
labels = data_df[data_df.columns[-1]]

# features_df: The feature dataset (independent variables)
features_df = data_df.drop(data_df.columns[-1], axis = 1)
features_df.head()

# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    features_df,        # Feature dataset
    labels,             # Target labels
    test_size=TEST_SIZE, # Fraction of data for testing (e.g., 0.2)
    stratify=labels,    # Maintain class balance between train and test sets
    random_state=RANDOM_STATE # Seed for reproducibility
)

# Print shapes of train and test datasets
X_train.shape, X_test.shape, y_train.shape, y_test.shape

# Initialize the StandardScaler
# StandardScaler standardizes features by removing the mean and scaling to unit variance.
scaler = StandardScaler()

# Fit the scaler on the training set and transform it
X_train = scaler.fit_transform(X_train)

# Transform the test set using the parameters learned from the training set
X_test = scaler.transform(X_test)

# Initialize the LabelEncoder
# LabelEncoder converts categorical labels into numerical form (e.g., 'A', 'B', 'C' -> 0, 1, 2)
le = LabelEncoder()

# Fit the encoder on the training labels and transform them
y_train = le.fit_transform(y_train)

# Transform the test labels using the same mapping from the training labels
y_test = le.transform(y_test)

# Verify the types of the transformed datasets
type(X_train), type(X_test), type(y_train), type(y_test)

# Define a sequential neural network model
model = nn.Sequential(
    nn.Linear(13, 8),  # Input layer: 13 input features, 8 neurons in the first hidden layer
    nn.ReLU(),         # Activation function: ReLU adds non-linearity
    nn.Linear(8, 3)    # Output layer: 3 neurons for 3 output classes (or values for regression)
)

# display the model architecture
display(model)

# Move to GPU
model = model.to(device)


summary(model, (13,))

# Define the loss function
# CrossEntropyLoss is suitable for multi-class classification problems.
# It combines nn.LogSoftmax() and nn.NLLLoss() in one single step for efficiency.
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer
# Adam is an adaptive optimization algorithm that adjusts learning rates dynamically.
optimizer = torch.optim.Adam(
    model.parameters(),  # Pass the model parameters to optimize
    lr=0.001             # Learning rate (default: 0.001, adjust as needed)
)
train_x = torch.tensor(X_train, dtype= torch.float32, device=device)
train_y = torch.tensor(y_train, dtype= torch.int64, device=device) 
# CrossEntropyLoss expects class indices as int64, not one-hot encoded

test_x = torch.tensor(X_test, dtype= torch.float32, device=device)
test_y = torch.tensor(y_test, dtype= torch.int64, device=device)

for epoch in range(EPOCHS):
    model.train()  # Set the model to training mode
    
    # Forward pass: Compute model predictions for the training data
    outputs = model(train_x)  # train_x should be moved to the correct device if using GPU/CPU dynamically
    
    # Compute the loss between predictions and true labels
    loss = loss_fn(outputs, train_y)  # Ensure train_y and outputs have compatible shapes and device

    optimizer.zero_grad()  # Reset gradients to avoid accumulation from previous steps
    loss.backward()        # Backpropagate to compute gradients
    
    optimizer.step()       # Update the model parameters using the optimizer
    
    if epoch % 100 == 0 or epoch == EPOCHS - 1:
        print(f'At epoch {epoch:4d} | Loss: {loss.item():.4f}')  # Use loss.item() to extract the scalar value


# Set the model to evaluation mode (important for layers like dropout or batch norm)
model.eval()

# Disable gradient computation for inference to improve performance and save memory
with torch.inference_mode():
    
    # Make predictions on the training data
    train_pred = model(train_x)

    # Make predictions on the testing data
    test_pred = model(test_x)

# Check the types of the predictions
type(train_pred), type(test_pred)


y_train_pred = train_pred.detach()  # Remove the computation graph (no gradients)
y_train_pred = y_train_pred.cpu().numpy()  # Convert tensor to NumPy array
y_train_pred = y_train_pred.argmax(axis=1)  # Convert logits to predicted class labels
y_train_pred

accuracy_score(y_train, y_train_pred)

print(classification_report(y_train, y_train_pred))

y_test_pred = test_pred.detach().cpu().numpy().argmax(axis = 1) # becomes np.ndarray of shape m x 1
y_test_pred

accuracy_score(y_test, y_test_pred)

print(classification_report(y_test, y_test_pred))


# Initialize lists to store the loss values for later analysis (e.g., plotting)
test_loss, train_loss = [], []

for epoch in range(EPOCHS):
    # Set the model to training mode
    model.train()
    
    # Forward pass: Compute model predictions for the training data
    outputs = model(train_x)  # train_x is the input training data
    loss = loss_fn(outputs, train_y)  # Compute the loss using the predicted outputs and true labels (train_y)

    # Back-propagation: Zero gradients, compute new gradients, and update model parameters
    optimizer.zero_grad()  # Clear the gradients from the previous step
    loss.backward()        # Back-propagate to compute gradients
    optimizer.step()       # Update the model parameters using the optimizer

    # Store the training loss for this epoch
    train_loss.append(loss.item())  # loss.item() returns the scalar value of the loss, not a tensor

    # No gradient computation needed during evaluation, so we disable it
    with torch.inference_mode():  # Disable gradients for the validation phase (saves memory and computation)
        
        # Set the model to evaluation mode (important for layers like dropout or batch norm)
        model.eval()

        # Forward pass: Compute model predictions for the test data
        outputs = model(test_x)  # test_x is the input test data
        tloss = loss_fn(outputs, test_y)  # Compute the test loss using the predicted outputs and true test labels

        # Store the test loss for this epoch
        test_loss.append(tloss.item())  # store test loss in list

    # Print progress at each epoch
    if epoch % 100 == 0 or epoch == EPOCHS - 1:
        print(f'At epoch {epoch:3d} | Train Loss: {loss.item():.4f} | Test Loss: {tloss.item():.4f}')


fig, ax = plt.subplots()

ax.plot(train_loss, label='Train')
ax.plot(test_loss, label='Test');


In [ ]:

# Neural Network (Keras) - Titanic Survival Prediction
#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)


df = pd.read_csv("titanic.csv")

df = df[[
    "survived",
    "pclass",
    "gender",
    "age",
    "fare"
]].dropna()


df["gender"] = df["gender"].map({
    "male": 0,
    "female": 1
})



X = df[[
    "pclass",
    "gender",
    "age",
    "fare"
]].values

y = df["survived"].values


X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,
    random_state=42
)




scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

# IMPORTANT:
# Use SAME scaler on test data
X_test = scaler.transform(X_test)


model = keras.Sequential([

    # Hidden Layer 1
    layers.Dense(
        8,
        activation='relu',
        input_shape=(4,)
    ),

    # Dropout
    layers.Dropout(0.3),

    # Hidden Layer 2
    layers.Dense(
        4,
        activation='relu'
    ),

    # Output Layer
    layers.Dense(
        1,
        activation='sigmoid'
    )
])


model.compile(

    optimizer='adam',

    loss='binary_crossentropy',

    metrics=['accuracy']
)


# STEP 8: Callbacks


# EarlyStopping:
# Stop training if validation loss
# stops improving

early_stop = EarlyStopping(

    monitor='val_loss',

    patience=10,

    restore_best_weights=True
)


# ModelCheckpoint:
# Save best model during training

checkpoint = ModelCheckpoint(

    "best_model.keras",

    monitor='val_loss',

    save_best_only=True,

    verbose=1
)


# -----------------------------------------------------
# STEP 9: Train Model
# -----------------------------------------------------
history = model.fit(

    X_train,
    y_train,

    epochs=100,

    batch_size=16,

    validation_split=0.2,

    callbacks=[
        early_stop,
        checkpoint
    ]
)



model.summary()



print("\n--- Test Evaluation ---")

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss     :", round(loss, 4))
print("Test Accuracy :", round(accuracy, 4))



test = scaler.transform(test)

p = model.predict(test)

# STEP 13: Training History
print("\n--- Final Metrics ---")

print(
    "Final Training Accuracy :",
    round(history.history['accuracy'][-1], 4)
)

print(
    "Final Validation Accuracy :",
    round(history.history['val_accuracy'][-1], 4)
)


# STEP 14: Plot Accuracy Graph
plt.figure(figsize=(8, 5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend()

plt.show()


# STEP 15: Plot Loss Graph
plt.figure(figsize=(8, 5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()